In [367]:
import pandas as pd

In [368]:
df_inicio = pd.read_csv("DATA\oficial.csv")

<>:1: SyntaxWarning: invalid escape sequence '\o'
<>:1: SyntaxWarning: invalid escape sequence '\o'
C:\Users\pedro\AppData\Local\Temp\ipykernel_25468\644740778.py:1: SyntaxWarning: invalid escape sequence '\o'
  df_inicio = pd.read_csv("DATA\oficial.csv")


In [369]:
df_inicio.head()

,temporada_atual,rodada_atual,id_piloto_atual,posicao_quali_atual,q1_atual,q2_atual,q3_atual,target,pontos_anterior_individual,id_circuito_atual,...,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,media_posicao_ganha_anterior,tendencia_desempenho,temp_ar_media,temp_pista_media,umidade_media,corrida_molhada,perc_voltas_chuva
0,2018,1,alonso,11.0,83597.0,83692.0,NaN,5,10.0,albert_park,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
1,2018,1,bottas,10.0,83686.0,82089.0,NaN,8,4.0,albert_park,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
2,2018,1,brendon_hartley,16.0,84532.0,NaN,NaN,15,0.0,albert_park,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
3,2018,1,ericsson,17.0,84556.0,NaN,NaN,19,0.0,albert_park,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5
4,2018,1,gasly,20.0,85295.0,NaN,NaN,18,0.0,albert_park,...,NaN,NaN,NaN,NaN,NaN,24.08,36.32,30.92,0,4.5


In [370]:
df_inicio.columns

Index(['temporada_atual', 'rodada_atual', 'id_piloto_atual',
       'posicao_quali_atual', 'q1_atual', 'q2_atual', 'q3_atual', 'target',
       'pontos_anterior_individual', 'id_circuito_atual', 'id_equipe_atual',
       'grid_anterior', 'posicao_ultima_corrida', 'status',
       'posicao_equipe_anterior', 'pontos_equipe_anterior',
       'vitorias_equipe_anterior', 'dif_para_pole_atual', 'pontos_anterior',
       'posicao_camp_anterior', 'num_vitorias_anterior',
       'media_ultimas_3_anterior', 'media_ultimas_5_anterior',
       'qtde_abandonos_anterior', 'media_posicao_ganha_anterior',
       'tendencia_desempenho', 'temp_ar_media', 'temp_pista_media',
       'umidade_media', 'corrida_molhada', 'perc_voltas_chuva'],
      dtype='object')

In [371]:
from datetime import datetime
import requests

ANO_ATUAL = datetime.now().year
proxima_rodada = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL)]["rodada_atual"].max()
pilotos = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] == proxima_rodada)]["id_piloto_atual"]
circuito = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada + 1}/circuits/").json()["MRData"]["CircuitTable"]["Circuits"][0]["circuitId"]
equipe = df_inicio[(df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["rodada_atual"] == proxima_rodada)]["id_equipe_atual"]

grid = []
infos_equipe = []
infos_piloto = []

acesso_results = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/results/").json()
results = acesso_results["MRData"]["RaceTable"]["Races"][0]["Results"]
for result in results:
    grid_1 = result["grid"]
    try:
        grid_1 = int(grid_1)
    except:
        grid_1 = 0
    grid_2 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == result["Driver"]["driverId"])]["grid_anterior"]
    grid_2 = pd.to_numeric(grid_2, errors="coerce").tail(2).sum()
    soma_grid = grid_2 + grid_1

    posicao_1 = result["position"]
    try:
        posicao_1 = int(posicao_1)
    except:
        posicao_1 = 0
    posicao_2 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == result["Driver"]["driverId"])]["posicao_ultima_corrida"]
    posicao_2 = pd.to_numeric(posicao_2, errors="coerce").tail(2).sum()
    soma_posicao = posicao_1 + posicao_2
    
    
    grid.append({
        "temporada_atual": int(acesso_results["MRData"]["RaceTable"]["season"]),
        "rodada_atual": int(acesso_results["MRData"]["RaceTable"]["round"]),
        "id_piloto_atual": result["Driver"]["driverId"],
        "grid_anterior": result["grid"],
        "posicao_ultima_corrida": int(result["position"]),
        "count_abandono": result["positionText"],
        "pontos_anterior_individual": result["points"],
        "media_posicao_ganha_anterior": f"{(soma_grid - soma_posicao)/3:.2f}",
        "status": result["status"],
    })
df_grid = pd.DataFrame(grid)
df_grid["media_posicao_ganha_anterior"] = df_grid["media_posicao_ganha_anterior"].astype(float)

acesso_equipe = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada}/constructorstandings/").json()
teams = acesso_equipe["MRData"]["StandingsTable"]["StandingsLists"][0]["ConstructorStandings"]
for team in teams:
    infos_equipe.append({
        "temporada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["season"]),
        "rodada_atual": int(acesso_equipe["MRData"]["StandingsTable"]["round"]),
        "id_equipe_atual": team["Constructor"]["constructorId"],
        "posicao_equipe_anterior": team["position"],
        "pontos_equipe_anterior": team["points"],
        "vitorias_equipe_anterior": team["wins"]
    })
df_team = pd.DataFrame(infos_equipe)

acesso_piloto = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/driverstandings/").json()
drivers = acesso_piloto["MRData"]["StandingsTable"]["StandingsLists"][0]["DriverStandings"]
for driver in drivers:
    df_posicao_1 = df_grid[(df_grid["rodada_atual"] == proxima_rodada) & (df_grid["temporada_atual"] == ANO_ATUAL) & (df_grid["id_piloto_atual"] == driver["Driver"]["driverId"])]["posicao_ultima_corrida"]
    df_posicao_2 = df_inicio[(df_inicio["rodada_atual"] <= proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == driver["Driver"]["driverId"])]["posicao_ultima_corrida"]

    abandonos = df_inicio[(df_inicio["rodada_atual"] == proxima_rodada) & (df_inicio["temporada_atual"] == ANO_ATUAL) & (df_inicio["id_piloto_atual"] == driver["Driver"]["driverId"])]["qtde_abandonos_anterior"].tail(1)
    infos_piloto.append({
        "temporada_atual": int(acesso_piloto["MRData"]["StandingsTable"]["season"]),
        "rodada_atual": int(acesso_piloto["MRData"]["StandingsTable"]["round"]),
        "id_piloto_atual": driver["Driver"]["driverId"],
        "pontos_anterior": driver["points"],
        "posicao_camp_anterior": driver["position"],
        "num_vitorias_anterior": driver["wins"],
        "media_ultimas_3_anterior": f"{pd.concat([pd.to_numeric(df_posicao_2, errors="coerce").tail(2).reset_index(drop=True), pd.to_numeric(df_posicao_1, errors="coerce").tail(1).reset_index(drop=True)]).mean():.2f}",
        "media_ultimas_5_anterior": f"{pd.concat([pd.to_numeric(df_posicao_2, errors="coerce").tail(4).reset_index(drop=True), pd.to_numeric(df_posicao_1, errors="coerce").tail(1).reset_index(drop=True)]).mean():.2f}",
        "qtde_abandonos_anterior": int(abandonos) + int(df_grid[(df_grid["rodada_atual"] == proxima_rodada) & (df_grid["temporada_atual"] == ANO_ATUAL) & (df_grid["id_piloto_atual"] == driver["Driver"]["driverId"]) & (df_grid["count_abandono"] == "R")].shape[0])
    })
df_driver = pd.DataFrame(infos_piloto)
df_driver["media_ultimas_3_anterior"] = df_driver["media_ultimas_3_anterior"].astype(float)
df_driver["media_ultimas_5_anterior"] = df_driver["media_ultimas_5_anterior"].astype(float)
df_driver["tendencia_desempenho"] = (df_driver["media_ultimas_3_anterior"] - df_driver["media_ultimas_5_anterior"]).round(2)

df = pd.DataFrame({
    'temporada_atual': ANO_ATUAL,
    'rodada_atual': proxima_rodada,
    'id_piloto_atual': pilotos,
    "id_circuito_atual": circuito,
    'id_equipe_atual': equipe,
})

C:\Users\pedro\AppData\Local\Temp\ipykernel_25468\3868084195.py:79: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  "qtde_abandonos_anterior": int(abandonos) + int(df_grid[(df_grid["rodada_atual"] == proxima_rodada) & (df_grid["temporada_atual"] == ANO_ATUAL) & (df_grid["id_piloto_atual"] == driver["Driver"]["driverId"]) & (df_grid["count_abandono"] == "R")].shape[0])
C:\Users\pedro\AppData\Local\Temp\ipykernel_25468\3868084195.py:79: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  "qtde_abandonos_anterior": int(abandonos) + int(df_grid[(df_grid["rodada_atual"] == proxima_rodada) & (df_grid["temporada_atual"] == ANO_ATUAL) & (df_grid["id_piloto_atual"] == driver["Driver"]["driverId"]) & (df_grid["count_abandono"] == "R")].shape[0])
C:\Users\pedro\AppData\Local\Temp\ipykernel_25468\3868084195.py:79: Futu

In [372]:
df_merge1 = pd.merge(
    df,
    df_grid,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

df_merge1.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,status
0,2026,6,albon,catalunya,williams,11,8,8,4,1.67,Finished
1,2026,6,alonso,catalunya,aston_martin,21,10,10,1,4.00,Finished
2,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,Finished
3,2026,6,arvid_lindblad,catalunya,rb,15,7,7,6,-1.00,Finished
4,2026,6,bearman,catalunya,haas,19,20,R,0,2.00,Retired


In [373]:
df_merge2 = pd.merge(
    df_merge1,
    df_team,
    on=["temporada_atual", "rodada_atual", "id_equipe_atual"],
    how="left"
)

df_merge2.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,status,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior
0,2026,6,albon,catalunya,williams,11,8,8,4,1.67,Finished,8,11,0
1,2026,6,alonso,catalunya,aston_martin,21,10,10,1,4.00,Finished,10,1,0
2,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,Finished,1,244,6
3,2026,6,arvid_lindblad,catalunya,rb,15,7,7,6,-1.00,Finished,6,35,0
4,2026,6,bearman,catalunya,haas,19,20,R,0,2.00,Retired,7,21,0


In [374]:
df_merge3 = pd.merge(
    df_merge2,
    df_driver,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

df_merge3.head()

,temporada_atual,rodada_atual,id_piloto_atual,id_circuito_atual,id_equipe_atual,grid_anterior,posicao_ultima_corrida,count_abandono,pontos_anterior_individual,media_posicao_ganha_anterior,...,posicao_equipe_anterior,pontos_equipe_anterior,vitorias_equipe_anterior,pontos_anterior,posicao_camp_anterior,num_vitorias_anterior,media_ultimas_3_anterior,media_ultimas_5_anterior,qtde_abandonos_anterior,tendencia_desempenho
0,2026,6,albon,catalunya,williams,11,8,8,4,1.67,...,8,11,0,5,15,0,13.00,16.2,0,-3.20
1,2026,6,alonso,catalunya,aston_martin,21,10,10,1,4.00,...,10,1,0,1,18,0,15.00,16.0,2,-1.00
2,2026,6,antonelli,catalunya,mercedes,1,1,1,25,0.33,...,1,244,6,156,1,5,1.00,1.0,0,0.00
3,2026,6,arvid_lindblad,catalunya,rb,15,7,7,6,-1.00,...,6,35,0,11,13,0,14.33,13.8,0,0.53
4,2026,6,bearman,catalunya,haas,19,20,R,0,2.00,...,7,21,0,18,11,0,13.67,13.6,2,0.07


In [375]:
df_merge3.drop(columns=["count_abandono"], inplace=True)

In [376]:
import requests
from datetime import datetime, timezone

acesso_circuito = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/circuits/").json()
latitude = acesso_circuito["MRData"]["CircuitTable"]["Circuits"][0]["Location"]["lat"]
longetude = acesso_circuito["MRData"]["CircuitTable"]["Circuits"][0]["Location"]["long"]

data_corrida = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/").json()
hora = data_corrida["MRData"]["RaceTable"]["Races"][0]["time"]
data = data_corrida["MRData"]["RaceTable"]["Races"][0]["date"]

acesso_clima = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longetude}&start_date={data}&end_date={data}&hourly=temperature_2m,relative_humidity_2m,precipitation,shortwave_radiation,surface_temperature&timezone=auto").json()

dt = datetime.strptime(f"{data}T{hora}", "%Y-%m-%dT%H:%M:%SZ")
dt = dt.replace(tzinfo=timezone.utc)
openmeteo_time = dt.strftime("%Y-%m-%dT%H:%M")

indice_clima = 0
hourly = acesso_clima["hourly"]["time"]
for indice, valor in enumerate(hourly):
    if valor == openmeteo_time:
        indice_clima = indice
        break

temp_ar_media_api = acesso_clima["hourly"]["temperature_2m"][indice_clima:indice_clima+3]
temp_pista_media_api = acesso_clima["hourly"]["surface_temperature"][indice_clima:indice_clima+3]
umidade_media_api = acesso_clima["hourly"]["relative_humidity_2m"][indice_clima:indice_clima+3]
precipitation_api = acesso_clima["hourly"]["precipitation"][indice_clima:indice_clima+3]

temp_ar_media = f"{sum(temp_ar_media_api) / len(temp_ar_media_api):.2f}"
temp_pista_media = f"{sum(temp_pista_media_api) / len(temp_pista_media_api):.1f}"
umidade_media = f"{sum(umidade_media_api) / len(umidade_media_api):.2f}"
corrida_molhada = int(sum(precipitation_api) > 0)
perc_voltas_chuva = f"{len([x for x in precipitation_api if x > 0]) / len(precipitation_api) * 100:.2f}"

df_merge3["temp_ar_media"] = float(temp_ar_media)
df_merge3["temp_pista_media"] = float(temp_pista_media)
df_merge3["umidade_media"] = float(umidade_media)
df_merge3["corrida_molhada"] = corrida_molhada
df_merge3["perc_voltas_chuva"] = float(perc_voltas_chuva)

In [377]:
acesso_results = requests.get(f"https://api.jolpi.ca/ergast/f1/{ANO_ATUAL}/{proxima_rodada+1}/qualifying/").json()
quali_results = acesso_results["MRData"]["RaceTable"]["Races"]

def tempo_para_ms(tempo_str):
    if pd.isna(tempo_str):
        return None
    try:
        minutos, resto = tempo_str.split(':')
        segundos, ms = resto.split('.')
        return int(minutos) * 60000 + int(segundos) * 1000 + int(ms)
    except:
        return None

if not quali_results:
    pass
else:
    quali_temp = acesso_results["MRData"]["RaceTable"]["Races"][0]

    quali_results = acesso_results["MRData"]["RaceTable"]["Races"][0]["QualifyingResults"]
    info_quali = []
    
    for quali in quali_results:
        info_quali.append({
            "posicao_quali_atual": int(quali["position"]),
            "q1_atual": quali.get("Q1", None),
            "q2_atual": quali.get("Q2", None),
            "q3_atual": quali.get("Q3", None),
            "id_piloto_atual": quali["Driver"]["driverId"],
            "rodada_atual": int(quali_temp["round"]),
            "temporada_atual": int(quali_temp["season"])
        })
    df_quali = pd.DataFrame(info_quali)

    df_quali['q1_atual'] = df_quali['q1_atual'].apply(tempo_para_ms)
    df_quali['q2_atual'] = df_quali['q2_atual'].apply(tempo_para_ms)
    df_quali['q3_atual'] = df_quali['q3_atual'].apply(tempo_para_ms)

    menor_tempo = df_quali["q3_atual"].min()

    q3_rodada = []
    for value_q3 in df_quali['q3_atual']:
        if pd.notna(value_q3) and value_q3 != '':
            dif = value_q3 - menor_tempo
            q3_rodada.append(dif)
        else:
            q3_rodada.append(None)

    df_quali["dif_para_pole_atual"] = q3_rodada 

    df_merge3 = pd.merge(
        df_merge3,
        df_quali,
        on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
        how="left"
    )
  

In [378]:
df_merge3["rodada_atual"] = proxima_rodada + 1

df_merge3.to_csv(r"DATA\analise.csv", index=False)